In [1]:
%pip install transformers torch librosa soundfile  datasets


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# mount google drive to fetch the relevant files

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ModuleNotFoundError: No module named 'google'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# This downloads the official files automatically
processor = WhisperProcessor.from_pretrained("openai/whisper-small.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small.en")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from transformers import WhisperForConditionalGeneration, WhisperProcessor, WhisperFeatureExtractor
import librosa
import torch

# FIX: Changed from a local path to the official model ID
# This will download the necessary files automatically
model_id = "openai/whisper-small.en"

# Load model
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Initialize processor and feature extractor
processor = WhisperProcessor.from_pretrained(model_id, language='English', task="transcribe")
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)

print("Model and Processor loaded successfully!")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model and Processor loaded successfully!


In [ ]:
import librosa
import torch

audio_file_path = "/content/drive/MyDrive/english speech.mp3"

try:
    # 1. Load audio file
    # sr=16000 is required for Whisper
    audio, sample_rate = librosa.load(audio_file_path, sr=16000)

    # 2. Extract and Preprocess features
    # Calling processor() directly handles padding and tensor conversion automatically
    inputs = processor(audio, sampling_rate=sample_rate, return_tensors="pt")
    input_features = inputs.input_features

    # 3. Perform inference
    with torch.no_grad():
        generated_ids = model.generate(input_features)

    # 4. Decode transcribed text
    transcribed_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    print("-" * 30)
    print("✅ Transcribed Text:", transcribed_text)
    print("-" * 30)

except FileNotFoundError:
    print(f"❌ Error: The file '{audio_file_path}' was not found.")
    print("Please upload your audio file to the folder icon on the left sidebar.")

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 

------------------------------
✅ Transcribed Text:  Prime Minister Modi and distinguished leaders, it's wonderful to be back in India. Every time I visit, I'm struck by the pace of change and today is no different. Back when I was a student, I often took the Coromandel Express train from Chennai up to IID Kharagpur. To get there, we passed through Vishakapatnam, Wysack. I remember it being a quiet and modest coastal city brimming with potential. Now in that same city,
------------------------------


In [ ]:
transcription= " ".join(transcribed_text)

In [ ]:
# 1. Uninstall current versions to avoid conflicts
#!pip uninstall -y torch torchaudio

# 2. Install compatible versions
#!pip install torch==2.1.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121
#!pip install speechbrain

In [ ]:
# 1. Install the latest version directly from their development branch
!pip install git+https://github.com/speechbrain/speechbrain.git

# 2. You must restart the session after this:
# Go to Runtime -> Restart session

  Cloning https://github.com/speechbrain/speechbrain.git to /tmp/pip-req-build-fk3sh_zc
  Running command git clone --filter=blob:none --quiet https://github.com/speechbrain/speechbrain.git /tmp/pip-req-build-fk3sh_zc
  Resolved https://github.com/speechbrain/speechbrain.git to commit aca5e41a34b3bcbe532640f2d3d0f6de3fd50ebe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# Modern TTS that actually works - no broken repos
!pip install -q kokoro>=0.9.4 soundfile

from kokoro import KPipeline
import soundfile as sf
from IPython.display import Audio

# Load pipeline (English)
pipeline = KPipeline(lang_code='a')  # 'a' = American English

# Use your Whisper transcription
transcription = transcribed_text

print(f"Generating speech for: {transcription[:80]}...")

# Generate audio
generator = pipeline(transcription, voice='af_heart', speed=1.0)

# Collect and save
import numpy as np
audio_chunks = [chunk for _, _, chunk in generator]
full_audio = np.concatenate(audio_chunks)

sf.write("generated_speech.wav", full_audio, 24000)
print("✅ Speech generated successfully!")
Audio("generated_speech.wav")

Generating speech for:  Prime Minister Modi and distinguished leaders, it's wonderful to be back in Ind...


voices/af_heart.pt:   0%|          | 0.00/523k [00:00<?, ?B/s]

✅ Speech generated successfully!
